In [2]:
import pandas as pd
import os 
import sys

# Custom imports from utils.py
script_dir = os.path.dirname(os.getcwd())
repo_path = script_dir.split("WP2_Sweet_Protein_Discovery")[0] + "/WP2_Sweet_Protein_Discovery"
sys.path.append(repo_path)
from utils import make_of3_json, fasta_to_dict

rewrite = False

# Find cluster files from mmseqs2 results
cluster_dir = f"{repo_path}/6_structural_analyses/2_mmseqs/clustered_hits"
cluster_dirs = [os.path.join(cluster_dir, d) for d in os.listdir(cluster_dir) if os.path.isdir(os.path.join(cluster_dir, d))]
cluster_files = []
for d in cluster_dirs:
    for f in os.listdir(d):
        if f.endswith("_cluster.tsv"):
            cluster_files.append(os.path.join(d, f))

# Read tm_score results for OneKP and UniProt
filtered_tm_scores_df = pd.read_csv(f"{repo_path}/6_structural_analyses/1_tm_scores/tm_scores_filtered.tsv", sep="\t")
for index, row in filtered_tm_scores_df.iterrows():
    if row["source"] == "UniProt":
        filtered_tm_scores_df.at[index, 'pdb'] = row["query"] + "___" + row['target'].split('|')[1]
    elif row["source"] == "OneKP":
        filtered_tm_scores_df.at[index, 'pdb'] = row["query"] + "___" + row['target']

# Remove high identity matches (pident >= 90%) to avoid redundancy
print(f"Number of rows before removing high identity matches: {len(filtered_tm_scores_df)}")
filtered_tm_scores_df = filtered_tm_scores_df[filtered_tm_scores_df['pident'] < 90]
print(f"Number of rows after removing high identity matches: {len(filtered_tm_scores_df)}")

# Read cropped fasta sequences
fasta_dir = f"{repo_path}/6_structural_analyses/1_tm_scores/merged_hits"
fasta_dfs = []
for sweet_protein in filtered_tm_scores_df["query"].unique():
    fasta_file = os.path.join(fasta_dir, f"{sweet_protein}_merged_hits.fasta")
    if os.path.exists(fasta_file):
        fasta_dict = fasta_to_dict(fasta_file)
        # Make into a df 
        fasta_df = pd.DataFrame(list(fasta_dict.items()), columns=["pdb", "cropped_tseq"])
        fasta_dfs.append(fasta_df)
fasta_df = pd.concat(fasta_dfs, ignore_index=True)

# Merge tm_score results with fasta sequences
filtered_tm_scores_df = pd.merge(filtered_tm_scores_df, fasta_df, on="pdb", how="left")

Number of rows before removing high identity matches: 10015
Number of rows after removing high identity matches: 10008


In [3]:
# Load Thaumatin I and II cluster information
thaumatin_clusters = []
for cluster_file in cluster_files:
    df = pd.read_csv(cluster_file, sep="\t", header=None, names=["cluster_id", "pdb"])
    thaumatin_clusters.append(df)

# Concatenate all cluster dataframes into one
thaumatin_cluster_df = pd.concat(thaumatin_clusters, ignore_index=True)
thaumatin_cluster_df = thaumatin_cluster_df.drop_duplicates()

# Merge tm_score results with cluster information
thaumatin_df = pd.merge(filtered_tm_scores_df, thaumatin_cluster_df, on="pdb", how="inner")

# Get best Grantham score for each cluster 
thaumatin_df = thaumatin_df.loc[thaumatin_df.groupby("cluster_id")["grantham_score"].idxmin()]

# From filtered_tm_scores_df, drop Thaumatin hits
filtered_tm_scores_df = filtered_tm_scores_df[~filtered_tm_scores_df['query'].str.contains("Thaumatin")]

# Concatenate thaumatin_df with the rest of the filtered_tm_scores_df
representative_hits_df = pd.concat([filtered_tm_scores_df, thaumatin_df], ignore_index=True)

# Write out fastas for the representative hits
if rewrite:
    fasta_dir = f"{repo_path}/6_structural_analyses/3_representative_structures/fastas/peptides"
    os.makedirs(fasta_dir, exist_ok=True)
    for index, row in representative_hits_df.iterrows():
        fasta_file = os.path.join(fasta_dir, f"{row['pdb']}.fasta")
        with open(fasta_file, 'w') as f:
            f.write(f">{row['pdb']}\n{row['cropped_tseq']}\n")

# Make a dictionary to save the files for the best hits for each cluster
rep_structures_path = f"{repo_path}/6_structural_analyses/3_representative_structures/"
if not os.path.exists(rep_structures_path):
    os.makedirs(rep_structures_path)

# Save resulting dataframe in the rep_structures_path
if rewrite:
    representative_hits_df.to_csv(os.path.join(rep_structures_path, "representative_structures.csv"), index=False)

# Copy sweet protein fastas to the same folder
if rewrite:
    sweet_protein_fastas = os.listdir(f"{repo_path}/1_viridiplantae_prot_db/4_sweet_proteins/sweet_protein_fastas/")
    for fasta in sweet_protein_fastas:
        src_path = f"{repo_path}/1_viridiplantae_prot_db/4_sweet_proteins/sweet_protein_fastas/{fasta}"
        dst_path = os.path.join(fasta_dir, fasta)
        if not os.path.exists(dst_path):
            os.system(f"cp {src_path} {dst_path}")

if rewrite:
    receptor_seq_dict = {
        "ts1r2_human": "EPAENSDFYLPGDYLLGGLFSLHANMKGIVHLNFLQVPMCKEYEVKVIGYNLMQAMRFAVEEINNDSSLLPGVLLGYEIVDVCYISNNVQPVLYFLAHEDNLLPIQEDYSNYISRVVAVIGPDNSESVMTVANFLSLFLLPQITYSAISDELRDKVRFPALLRTTPSADHHIEAMVQLMLHFRWNWIIVLVSSDTYGRDNGQLLGERVARRDICIAFQETLPTLQPNQNMTSEERQRLVTIVDKLQQSTARVVVVFSPDLTLYHFFNEVLRQNFTGAVWIASESWAIDPVLHNLTELRHLGTFLGITIQSVPIPGFSEFREWGPQAGPPPLSRTSQSYTCNQECDNCLNATLSFNTILRLSGERVVYSVYSAVYAVAHALHSLLGCDKSTCTKRVVYPWQLLEEIWKVNFTLLDHQIFFDPQGDVALHLEIVQWQWDRSQNPFQSVASYYPLQRQLKNIQDISWHTINNTIPMSMCSKRCQSGQKKKPVGIHVCCFECIDCLPGTFLNHTEDEYECQACPNNEWSYQSETSCFKRQLVFLEWHEAPTIAVALLAALGFLSTLAILVIFWRHFQTPIVRSAGGPMCFLMLTLLLVAYMVVPVYVGPPKVSTCLCRQALFPLCFTICISCIAVRSFQIVCAFKMASRFPRAYSYWVRYQGPYVSMAFITVLKMVIVVIGMLATGLSPTTRTDPDDPKITIVSCNPNYRNSLLFNTSLDLLLSVVGFSFAYMGKELPTNYNEAKFITLSMTFYFTSSVSLCTFMSAYSGVLVTIVDLLVTVLNLLAISLGYFGPKCYMILFYPERNTPAYFNSMIQGYTMRRD",
        "ts1r3_human": "APLCLSQQLRMKGDYVLGGLFPLGEAEEAGLRSRTRPSSPVCTRFSSNGLLWALAMKMAVEEINNKSDLLPGLRLGYDLFDTCSEPVVAMKPSLMFLAKAGSRDIAAYCNYTQYQPRVLAVIGPHSSELAMVTGKFFSFFLMPQVSYGASMELLSARETFPSFFRTVPSDRVQLTAAAELLQEFGWNWVAALGSDDEYGRQGLSIFSALAAARGICIAHEGLVPLPRADDSRLGKVQDVLHQVNQSSVQVVLLFASVHAAHALFNYSISSRLSPKVWVASEAWLTSDLVMGLPGMAQMGTVLGFLQRGAQLHEFPQYVKTHLALATDPAFCSALGEREQGLEEDVVGQRCPQCDCITLQNVSAGLNHHQTFSVYAAVYSVAQALHNTLQCNASGCPAQDPVKPWQLLENMYNLTFHVGGLPLRFDSSGNVDMEYDLKLWVWQGSVPRLHDVGRFNGSLRTERLKIRWHTSDNQKPVSRCSRQCQEGQVRRVKGFHSCCYDCVDCEAGSYRQNPDDIACTFCGQDEWSPERSTRCFRRRSRFLAWGEPAVLLLLLLLSLALGLVLAALGLFVHHRDSPLVQASGGPLACFGLVCLGLVCLSVLLFPGQPSPARCLAQQPLSHLPLTGCLSTLFLQAAEIFVESELPLSWADRLSGCLRGPWAWLVVLLAMLVEVALCTWYLVAFPPEVVTDWHMLPTEALVHCRTRSWVSFGLAHATNATLAFLCFLGTFLVRSQPGCYNRARGLTFAMLAYFITWVSFVPLLANVQVVLRPAVQMGALLLCVLGILAAFHLPRCYLLMRQPGLNTPEFFLGGGPGDAQGQNDGNTGNQGKHE"
    }

    # Save receptor sequences to fasta files
    receptor_fasta_dir = f"{repo_path}/6_structural_analyses/3_representative_structures/fastas/receptors"
    os.makedirs(receptor_fasta_dir, exist_ok=True)
    for receptor, seq in receptor_seq_dict.items():
        fasta_file = os.path.join(receptor_fasta_dir, f"{receptor}.fasta")
        with open(fasta_file, 'w') as f:
            f.write(f">{receptor}\n{seq}\n")

# Make OpenFold 3 JSON files for complex modelling with the sweet taste receptor heterodimer

In [ ]:
msa_dir = "/dcai/projects01/cu_0027/sweet_protein_complexes"
local_msa_dir = "~/Desktop/msas/"
local_zip_dir = "~/Desktop/alignments/"
fasta_dir = f"{repo_path}/6_structural_analyses/3_representative_structures/fastas/peptides"
output_dir = f"{repo_path}/6_structural_analyses/3_representative_structures/of3_jsons"
monomer_json_dir = os.path.join(output_dir, "monomers")
complex_json_dir = os.path.join(output_dir, "complexes")
os.makedirs(monomer_json_dir, exist_ok=True)
os.makedirs(complex_json_dir, exist_ok=True)

for index, row in representative_hits_df.iterrows():
    sweet_protein_seq = row["cropped_tseq"]
    sweet_protein_name = row["target"]
    if row["source"] == "UniProt":
        sweet_protein_name = row["target"].split('|')[1]
    else:
        sweet_protein_name = row["target"]
    sweet_protein_name = row["query"] + "___" + sweet_protein_name
    sweet_protein_name = sweet_protein_name.replace("+", "_")
    chain_dict = {
        "A": {"seq": receptor_seq_dict["ts1r2_human"], "msa_path": f"{msa_dir}/alignments/ts1r2_human/uniref90_hits.a3m"}, 
        "B": {"seq":  receptor_seq_dict["ts1r3_human"], "msa_path": f"{msa_dir}/alignments/ts1r3_human/uniref90_hits.a3m"},
        "C": {"seq": sweet_protein_seq, "msa_path": f"{msa_dir}/alignments/{sweet_protein_name}/uniref90_hits.a3m"}    
    }

    for chain in chain_dict.keys():
        msa_path = chain_dict[chain]["msa_path"]
        if chain == "A":
            local_path = os.path.join(local_msa_dir, "receptor", "ts1r2_human.a3m")
        elif chain == "B":
            local_path = os.path.join(local_msa_dir, "receptor", "ts1r3_human.a3m")
        elif chain == "C":
            local_path = os.path.join(local_msa_dir, "peptide", f"{sweet_protein_name}.a3m")
        else:   
            print(f"Unknown chain {chain} for {sweet_protein_name}. Skipping MSA check for this chain.")
            continue

        if not os.path.exists(local_path):
            print(f"Warning: MSA file {local_path} does not exist for chain {chain} of {sweet_protein_name}. Skipping this entry.")
            continue
        else:
            if chain == "A":
                zip_subdir = os.path.join(local_zip_dir, "ts1r2_human")
            elif chain == "B":
                zip_subdir = os.path.join(local_zip_dir, "ts1r3_human")
            elif chain == "C":
                zip_subdir = os.path.join(local_zip_dir, sweet_protein_name)
                
            zipped_msa_path = os.path.join(zip_subdir, "uniref90_hits.a3m")
            if not os.path.exists(zipped_msa_path):
                os.makedirs(zip_subdir, exist_ok=True)
                os.system(f"cp {local_path} {zipped_msa_path}")

    make_of3_json(chain_dict, model_name=sweet_protein_name, num_recycles=3, json_path=f"{output_dir}/complexes/{sweet_protein_name}.json", use_paired_msas=False)

# Make OF3 input jsons for native sweet proteins
sweet_protein_dict = {}
for sweet_protein in representative_hits_df["query"].unique():
    fasta_file = os.path.join(fasta_dir, f"{sweet_protein}.fasta")
    fasta_dict = fasta_to_dict(fasta_file)
    sweet_protein_seq = list(fasta_dict.values())[0]
    sweet_protein_dict[sweet_protein] = sweet_protein_seq

for key, item in sweet_protein_dict.items():
    sweet_protein_seq = item
    sweet_protein_name = key
    chain_dict = {
        "A": {"seq": receptor_seq_dict["ts1r2_human"], "msa_path": f"{msa_dir}/alignments/ts1r2_human/uniref90_hits.a3m"}, 
        "B": {"seq":  receptor_seq_dict["ts1r3_human"], "msa_path": f"{msa_dir}/alignments/ts1r3_human/uniref90_hits.a3m"},
        "C": {"seq": sweet_protein_seq, "msa_path": f"{msa_dir}/alignments/{sweet_protein_name}/uniref90_hits.a3m"}    
    }

    for chain in chain_dict.keys():
        msa_path = chain_dict[chain]["msa_path"]
        if chain == "A":
            local_path = os.path.join(local_msa_dir, "receptor", "ts1r2_human.a3m")
        elif chain == "B":
            local_path = os.path.join(local_msa_dir, "receptor", "ts1r3_human.a3m")
        elif chain == "C":
            local_path = os.path.join(local_msa_dir, "peptide", f"{sweet_protein_name}.a3m")
        else:   
            print(f"Unknown chain {chain} for {sweet_protein_name}. Skipping MSA check for this chain.")
            continue

        if not os.path.exists(local_path):
            print(f"Warning: MSA file {local_path} does not exist for chain {chain} of {sweet_protein_name}. Skipping this entry.")
            continue
        else:
            # Copy MSA file to expected location if it doesn't exist there
            if chain == "A":
                zip_subdir = os.path.join(local_zip_dir, "ts1r2_human")
            elif chain == "B":
                zip_subdir = os.path.join(local_zip_dir, "ts1r3_human")
            elif chain == "C":
                zip_subdir = os.path.join(local_zip_dir, sweet_protein_name)
                
            # Check if the zipped MSA file exists
            zipped_msa_path = os.path.join(zip_subdir, "uniref90_hits.a3m")
            if not os.path.exists(zipped_msa_path):
                os.makedirs(zip_subdir, exist_ok=True)
                os.system(f"cp {local_path} {zipped_msa_path}")

    make_of3_json(chain_dict, model_name=sweet_protein_name, num_recycles=3, json_path=f"{output_dir}/complexes/{sweet_protein_name}.json", use_paired_msas=False)

# Make OF3 input jsons for remodelling cropped hits

In [ ]:
msa_dir = "/dcai/projects01/cu_0027/sweet_protein_monomers"
local_msa_dir = "~/Desktop/msas/peptide"
local_zip_dir = "~/Desktop/alignments/"
for index, row in representative_hits_df.iterrows():
    sweet_protein_seq = row["cropped_tseq"]
    sweet_protein_name = row["target"]
    if row["source"] == "UniProt":
        sweet_protein_name = row["target"].split('|')[1]
    else:
        sweet_protein_name = row["target"]

    sweet_protein_name = row["query"] + "___" + sweet_protein_name
    sweet_protein_name = sweet_protein_name.replace("+", "_")
    chain_dict = {
        "A": {"seq": sweet_protein_seq, "msa_path": f"{msa_dir}/alignments/{sweet_protein_name}/uniref90_hits.a3m"}    
    }

    for chain in chain_dict.keys():
        msa_path = chain_dict[chain]["msa_path"]
        if chain == "A":
            local_path = os.path.join(local_msa_dir, f"{sweet_protein_name}.a3m")
        else:   
            print(f"Unknown chain {chain} for {sweet_protein_name}. Skipping MSA check for this chain.")
            continue

        if not os.path.exists(local_path):
            print(f"Warning: MSA file {local_path} does not exist for chain {chain} of {sweet_protein_name}. Skipping this entry.")
            continue
        else:
            if chain == "A":
                zip_subdir = os.path.join(local_zip_dir, sweet_protein_name)
                
            # Check if the zipped MSA file exists
            zipped_msa_path = os.path.join(zip_subdir, "uniref90_hits.a3m")
            if not os.path.exists(zipped_msa_path):
                os.makedirs(zip_subdir, exist_ok=True)
                os.system(f"cp {local_path} {zipped_msa_path}")

    make_of3_json(chain_dict, model_name=sweet_protein_name, num_recycles=3, json_path=f"{output_dir}/monomers/{sweet_protein_name}.json", use_paired_msas=True)